# 02 — GLM-OCR LoRA Fine-tuning (Google Colab Pro)

This notebook follows the **official GLM-OCR + LLaMA-Factory** LoRA recipe and adapts only the data paths, validation set, checkpoint cadence, and Colab resource handling.

Official recipe: https://github.com/zai-org/GLM-OCR/blob/main/examples/finetune/README.md

Retained official settings: `template=glm_ocr`, `<image>Text Recognition:`, LoRA rank 8 / target `all`, LR `1e-4`, 3 epochs, cosine, warmup 0.1. The official recipe uses batch 4 × grad accumulation 4; this notebook preserves effective batch 16 while lowering micro-batch if required by the Colab GPU.

## 0. Install LLaMA-Factory using the official GLM-OCR sequence

In [ ]:
%pip install -q -U "kagglehub>=1.0.2" "jiwer>=4.0.0" pandas pillow

In [ ]:
%cd /content
!rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd /content/LLaMA-Factory
%pip install -q -e .
%pip install -q -r requirements/metrics.txt
%pip install -q -U "transformers>=5.3.0,<5.18"
!llamafactory-cli version

## 1. Load data

In [ ]:
import os, json, time, random, platform, unicodedata, gc, math, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from jiwer import cer, wer

from google.colab import drive
drive.mount('/content/drive')

# Optional Colab secret. KaggleHub can also prompt/authenticate through its normal flow.
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except Exception:
    pass

import kagglehub

SEED = 42
RAW_HANDLE = 'ntklinhfitus/uit-hwdb'
MANIFEST_HANDLE = 'ntklinhfitus/uit-hwdb-manifest'
PROJECT_ROOT = Path('/content/drive/MyDrive/vlm_handwriting_ocr')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Try a partial raw download first. Fall back to the full Kaggle dataset if the
# installed KaggleHub/runtime does not accept directory-level download.
try:
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE, path='UIT_HWDB_line'))
except Exception as e:
    print('Partial raw download unavailable, falling back to full dataset:', repr(e))
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE))
manifest_download = Path(kagglehub.dataset_download(MANIFEST_HANDLE))

def locate_raw_line_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    candidates=[p for p in pool if (p/'train_data').is_dir() and (p/'test_data').is_dir()]
    assert candidates, f'Cannot locate UIT-HWDB-line train_data/test_data under {base}'
    candidates.sort(key=lambda p: ('UIT_HWDB_line' not in str(p), len(str(p))))
    return candidates[0]

def locate_manifest_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    for p in pool:
        if all((p/f).exists() for f in ['train.csv','val.csv','test.csv']):
            return p
    raise FileNotFoundError(f'Cannot locate train.csv/val.csv/test.csv under {base}')

RAW_ROOT=locate_raw_line_root(raw_download)
MANIFEST_ROOT=locate_manifest_root(manifest_download)
print('RAW_ROOT      =',RAW_ROOT)
print('MANIFEST_ROOT =',MANIFEST_ROOT)

In [ ]:
train_df=pd.read_csv(MANIFEST_ROOT/'train.csv')
val_df=pd.read_csv(MANIFEST_ROOT/'val.csv')
test_df=pd.read_csv(MANIFEST_ROOT/'test.csv')
EXPECTED={'train':6346,'validation':682,'test':201}
assert len(train_df)==EXPECTED['train'],len(train_df)
assert len(val_df)==EXPECTED['validation'],len(val_df)
assert len(test_df)==EXPECTED['test'],len(test_df)
required={'writer_id','filename','relative_path','text'}
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=required-set(frame.columns)
    assert not missing,f'{name} missing columns: {missing}'
train_writers=set(train_df.writer_id); val_writers=set(val_df.writer_id); test_writers=set(test_df.writer_id)
assert train_writers.isdisjoint(val_writers)
assert train_writers.isdisjoint(test_writers)
assert val_writers.isdisjoint(test_writers)

def resolve_image_path(row):
    return RAW_ROOT/str(row['relative_path'])
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=[str(resolve_image_path(r)) for _,r in frame.iterrows() if not resolve_image_path(r).exists()]
    assert not missing,f'{name}: missing image paths, e.g. {missing[:3]}'
print(f'Train      : {len(train_df)} samples | {len(train_writers)} writers')
print(f'Validation : {len(val_df)} samples | {len(val_writers)} writers')
print(f'Test       : {len(test_df)} samples | {len(test_writers)} writers')
print('✅ Frozen writer-disjoint split verified.')

## 2. GPU-aware batch profile

In [ ]:
import torch
assert torch.cuda.is_available(),'GPU required.'
GPU_NAME=torch.cuda.get_device_name(0); VRAM_GB=torch.cuda.get_device_properties(0).total_memory/1024**3; BF16=torch.cuda.is_bf16_supported()
if VRAM_GB>=20: MICRO_BATCH,GRAD_ACC=4,4
elif VRAM_GB>=12: MICRO_BATCH,GRAD_ACC=2,8
else: MICRO_BATCH,GRAD_ACC=1,16
assert MICRO_BATCH*GRAD_ACC==16
print(f'GPU={GPU_NAME} | VRAM={VRAM_GB:.1f}GB | BF16={BF16} | micro={MICRO_BATCH} | grad_acc={GRAD_ACC}')

## 3. Convert train/validation to official ShareGPT multimodal format

In [ ]:
LLF_ROOT=Path('/content/LLaMA-Factory'); DATA_DIR=LLF_ROOT/'data'; DATA_DIR.mkdir(exist_ok=True)
IMAGE_LINK=DATA_DIR/'uit_hwdb_line_images'
if IMAGE_LINK.exists() or IMAGE_LINK.is_symlink():
    if IMAGE_LINK.is_dir() and not IMAGE_LINK.is_symlink(): shutil.rmtree(IMAGE_LINK)
    else: IMAGE_LINK.unlink()
IMAGE_LINK.symlink_to(RAW_ROOT,target_is_directory=True)

def to_sharegpt(frame):
    return [{'messages':[{'role':'user','content':'<image>Text Recognition:'},{'role':'assistant','content':str(r['text'])}], 'images':[f"uit_hwdb_line_images/{r['relative_path']}"]} for _,r in frame.iterrows()]
train_json=DATA_DIR/'uit_hwdb_line_train.json'; val_json=DATA_DIR/'uit_hwdb_line_val.json'
train_json.write_text(json.dumps(to_sharegpt(train_df),ensure_ascii=False),encoding='utf-8')
val_json.write_text(json.dumps(to_sharegpt(val_df),ensure_ascii=False),encoding='utf-8')
print('train=',len(train_df),'val=',len(val_df))

## 4. Register datasets in LLaMA-Factory

In [ ]:
info_path=DATA_DIR/'dataset_info.json'; info=json.loads(info_path.read_text(encoding='utf-8'))
def entry(file_name):
    return {'file_name':file_name,'formatting':'sharegpt','columns':{'messages':'messages','images':'images'},'tags':{'role_tag':'role','content_tag':'content','user_tag':'user','assistant_tag':'assistant'}}
info['uit_hwdb_line_train']=entry(train_json.name); info['uit_hwdb_line_val']=entry(val_json.name)
info_path.write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
print('registered')

## 5. Build the official-style LoRA config

In [ ]:
OUTPUT_DIR=PROJECT_ROOT/'checkpoints'/'glm_ocr_lora'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CONFIG=Path('/content/glm_ocr_uit_hwdb_lora.yaml')
precision_lines='bf16: true\nfp16: false' if BF16 else 'bf16: false\nfp16: true'
lines=[
'### model','model_name_or_path: zai-org/GLM-OCR','trust_remote_code: true','',
'### method','stage: sft','do_train: true','finetuning_type: lora','lora_rank: 8','lora_target: all','',
'### dataset','dataset: uit_hwdb_line_train','eval_dataset: uit_hwdb_line_val','template: glm_ocr','cutoff_len: 2048','preprocessing_num_workers: 8','dataloader_num_workers: 2','',
'### output',f'output_dir: {OUTPUT_DIR}','logging_steps: 10','save_strategy: epoch','plot_loss: true','overwrite_output_dir: true','save_only_model: false','report_to: none','',
'### train',f'per_device_train_batch_size: {MICRO_BATCH}',f'gradient_accumulation_steps: {GRAD_ACC}','learning_rate: 1.0e-4','num_train_epochs: 3.0','lr_scheduler_type: cosine','warmup_ratio: 0.1','seed: 42',precision_lines,'ddp_timeout: 180000000','',
'### eval','do_eval: true','per_device_eval_batch_size: 1','eval_strategy: epoch','predict_with_generate: false'
]
yaml_text='\n'.join(lines)+'\n'; CONFIG.write_text(yaml_text,encoding='utf-8'); print(CONFIG.read_text())

## 6. 500/100 one-epoch smoke training gate

In [ ]:
RUN_SMOKE_TRAINING=True
RUN_FULL_TRAINING=False
smoke_train=DATA_DIR/'uit_hwdb_line_train_smoke.json'; smoke_val=DATA_DIR/'uit_hwdb_line_val_smoke.json'
smoke_train.write_text(json.dumps(to_sharegpt(train_df.sample(n=500,random_state=SEED)),ensure_ascii=False),encoding='utf-8')
smoke_val.write_text(json.dumps(to_sharegpt(val_df.sample(n=100,random_state=SEED)),ensure_ascii=False),encoding='utf-8')
info=json.loads(info_path.read_text(encoding='utf-8')); info['uit_hwdb_line_train_smoke']=entry(smoke_train.name); info['uit_hwdb_line_val_smoke']=entry(smoke_val.name); info_path.write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
SMOKE_DIR=Path('/content/glm_ocr_smoke_adapter'); SMOKE_CONFIG=Path('/content/glm_ocr_smoke.yaml')
smoke_yaml=yaml_text.replace('dataset: uit_hwdb_line_train','dataset: uit_hwdb_line_train_smoke').replace('eval_dataset: uit_hwdb_line_val','eval_dataset: uit_hwdb_line_val_smoke').replace(f'output_dir: {OUTPUT_DIR}',f'output_dir: {SMOKE_DIR}').replace('num_train_epochs: 3.0','num_train_epochs: 1.0')
SMOKE_CONFIG.write_text(smoke_yaml,encoding='utf-8')
print('RUN_SMOKE_TRAINING=',RUN_SMOKE_TRAINING,'RUN_FULL_TRAINING=',RUN_FULL_TRAINING)

In [ ]:
if RUN_SMOKE_TRAINING:
    os.chdir('/content/LLaMA-Factory')
    rc=subprocess.run(['llamafactory-cli','train',str(SMOKE_CONFIG)],env={**os.environ,'DISABLE_VERSION_CHECK':'1','CUDA_VISIBLE_DEVICES':'0'}).returncode
    assert rc==0,f'Smoke training failed: {rc}'
    assert SMOKE_DIR.exists(); print('✅ Smoke training completed:',SMOKE_DIR)
else: print('Smoke skipped.')

## 7. Full 3-epoch LoRA training

In [ ]:
RUN_FULL_TRAINING=False  # manually change after smoke succeeds
if RUN_FULL_TRAINING:
    os.chdir('/content/LLaMA-Factory')
    rc=subprocess.run(['llamafactory-cli','train',str(CONFIG)],env={**os.environ,'DISABLE_VERSION_CHECK':'1','CUDA_VISIBLE_DEVICES':'0'}).returncode
    assert rc==0,f'Full training failed: {rc}'
    print('✅ Full training completed:',OUTPUT_DIR)
else: print('Full run gated.')

## 8. Checkpoint inventory and provenance

In [ ]:
checkpoints=sorted([p for p in OUTPUT_DIR.glob('checkpoint-*') if p.is_dir()],key=lambda p:int(p.name.split('-')[-1])) if OUTPUT_DIR.exists() else []
for p in checkpoints: print(' -',p)
prov={'model':'zai-org/GLM-OCR','method':'official LLaMA-Factory LoRA adaptation','lora_rank':8,'lora_target':'all','learning_rate':1e-4,'epochs':3,'micro_batch':MICRO_BATCH,'gradient_accumulation':GRAD_ACC,'effective_batch':16,'gpu':GPU_NAME,'vram_gb':VRAM_GB,'bf16':BF16,'seed':SEED,'train_samples':len(train_df),'val_samples':len(val_df)}
(PROJECT_ROOT/'checkpoints'/'glm_ocr_lora_training_provenance.json').write_text(json.dumps(prov,indent=2),encoding='utf-8')